In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from PIL import Image
import os

# 1. Load Image

In [ ]:
# Input path dan load gambar
image_paths = {
    "image_1": "/content/drive/MyDrive/PCD_Assignment01/images/image 1.jpeg",
    "image_2": "/content/drive/MyDrive/PCD_Assignment01/images/image 2.jpg",
}

images = {}

for name, path in image_paths.items():
    image = Image.open(path)
    images[name] = image

    print(f"{name}")
    print(f"Format : {image.format}")
    print(f"Mode   : {image.mode}")
    print(f"Size   : {image.size}")
    print()

# 2. Matrix Representation

In [ ]:
# Representasikan gambar dalam matriks

def image_to_matrix(image):
    width, height = image.size

    matrix = []

    for y in range(height):
        row = []

        for x in range(width):
            pixel = image.getpixel((x, y))
            row.append(pixel)

        matrix.append(row)

    return matrix


matrices = {}

for name, image in images.items():
    matrices[name] = image_to_matrix(image)

    height = len(matrices[name])
    width = len(matrices[name][0])

    print(f"{name}")
    print(f"Matrix size : {height} × {width}")
    print(f"First pixel : {matrices[name][0][0]}")
    print()

In [ ]:
# Siapkan fungsi konversi matriks ke image

def matrix_to_image(matrix):
    height = len(matrix)
    width = len(matrix[0])

    image = Image.new("RGB", (width, height))

    for y in range(height):
        for x in range(width):
            image.putpixel((x, y), matrix[y][x])

    return image

test_matrix = matrices["image_1"]
test_image = matrix_to_image(test_matrix)

print(test_image.size)
display(test_image)

# Sampling Class

In [ ]:
class Sampling():
  def __init__(self, gambar, matriks_gambar):
    self.gambar = gambar
    self.matriks_gambar = matriks_gambar

  def run_downsampling(self, nama_teknik, scale):
    output_folder = "/content/drive/MyDrive/PCD_Assignment01/results/downsampling"
    os.makedirs(output_folder, exist_ok=True)

    for name, image in self.gambar.items():
      detail_matrix = self.matriks_gambar[name]

      if nama_teknik.lower() == "average":
        detail_downsampling = average_downsampling(detail_matrix, scale)
      elif nama_teknik.lower() == "max":
        detail_downsampling = max_downsampling(detail_matrix, scale)
      elif nama_teknik.lower() == "median":
        detail_downsampling = median_downsampling(detail_matrix, scale)
      else:
        print("Teknik downsampling tidak valid!")
        return

      detail_downsampling_image = matrix_to_image(detail_downsampling)
      display(detail_downsampling_image)

      output_path = os.path.join(output_folder, f"{name}_{nama_teknik}.png")
      detail_downsampling_image.save(output_path)
      print(f"Saved Downsampled: {output_path}")

  def run_upsampling(self, nama_teknik, scale, source_name="", mode="min"):
    # Menentukan folder output berdasarkan sumber
    sub_folder = "from_original" if source_name == "original" else "from_downsampled"
    output_folder = f"/content/drive/MyDrive/PCD_Assignment01/results/upsampling/{sub_folder}"
    os.makedirs(output_folder, exist_ok=True)

    for name, image in self.gambar.items():
      detail_matrix = self.matriks_gambar[name]

      if nama_teknik.lower() == "nearest neighbor":
        detail_upsampling = nn_upsampling(detail_matrix, scale, mode)
      elif nama_teknik.lower() == "bilinear":
        detail_upsampling = bilinear_upsampling(detail_matrix, scale)
      elif nama_teknik.lower() == "bicubic":
        detail_upsampling = bicubic_upsampling(detail_matrix, scale)
      else:
        print("Teknik upsampling tidak valid!")
        return

      detail_upsampling_image = matrix_to_image(detail_upsampling)
      print(f"\n{name} ({source_name} source)")
      display(detail_upsampling_image)

      output_path = os.path.join(output_folder, f"{name}_{nama_teknik.replace(' ', '_')}.png")
      detail_upsampling_image.save(output_path)
      print(f"Saved Upsampled: {output_path}")

In [ ]:
# Faktor sampling
scale = 2

# 3. Downsampling Algorithm

In [ ]:
downsampling = Sampling(images, matrices)

## 3.1 Average downsampling

Definisi fungsi:

In [ ]:
def average_downsampling(matrix, scale):
    input_height = len(matrix)
    input_width = len(matrix[0])

    output_height = input_height // scale
    output_width = input_width // scale

    output_matrix = []

    for y_out in range(output_height):
        row = []

        for x_out in range(output_width):

            # Posisi awal blok pada gambar input
            y_start = y_out * scale
            x_start = x_out * scale

            # Menampung nilai R, G, dan B
            sum_r = 0
            sum_g = 0
            sum_b = 0

            # Menelusuri pixel dalam blok
            for y in range(y_start, y_start + scale):
                for x in range(x_start, x_start + scale):

                    r, g, b = matrix[y][x]

                    sum_r += r
                    sum_g += g
                    sum_b += b

            # Menghitung rata-rata
            number_of_pixels = scale * scale

            avg_r = round(sum_r / number_of_pixels)
            avg_g = round(sum_g / number_of_pixels)
            avg_b = round(sum_b / number_of_pixels)

            # Pixel hasil
            new_pixel = (avg_r, avg_g, avg_b)

            row.append(new_pixel)

        output_matrix.append(row)

    return output_matrix

Penerapan pada gambar:

In [ ]:
downsampling.run_downsampling("average", scale)

## 3.2 Max Downsampling

Definisi fungsi:

In [ ]:
def max_downsampling(matrix, scale):
    input_height = len(matrix)
    input_width = len(matrix[0])

    output_height = input_height // scale
    output_width = input_width // scale

    output_matrix = []

    for y_out in range(output_height):
        row = []

        for x_out in range(output_width):

            # Posisi awal blok pada gambar input
            y_start = y_out * scale
            x_start = x_out * scale

            # Nilai awal maksimum
            max_r = 0
            max_g = 0
            max_b = 0

            # Menelusuri pixel dalam blok
            for y in range(y_start, y_start + scale):
                for x in range(x_start, x_start + scale):

                    r, g, b = matrix[y][x]

                    # Cari nilai maksimum setiap channel
                    if r > max_r:
                        max_r = r

                    if g > max_g:
                        max_g = g

                    if b > max_b:
                        max_b = b

            # Pixel hasil
            new_pixel = (max_r, max_g, max_b)

            row.append(new_pixel)

        output_matrix.append(row)

    return output_matrix

Penerapan pada gambar:

In [ ]:
downsampling.run_downsampling("max", scale)

## 3.3 Median Downsampling

Definisi fungsi:

In [ ]:
def calculate_median(values):
    values = sorted(values)

    n = len(values)

    middle = n // 2

    if n % 2 == 1:
        return values[middle]

    else:
        return round((values[middle - 1] + values[middle]) / 2)

In [ ]:
def median_downsampling(matrix, scale):
    input_height = len(matrix)
    input_width = len(matrix[0])

    output_height = input_height // scale
    output_width = input_width // scale

    output_matrix = []

    for y_out in range(output_height):
        row = []

        for x_out in range(output_width):

            # Posisi awal blok pada gambar input
            y_start = y_out * scale
            x_start = x_out * scale

            # Menampung nilai setiap channel
            values_r = []
            values_g = []
            values_b = []

            # Menelusuri pixel dalam blok
            for y in range(y_start, y_start + scale):
                for x in range(x_start, x_start + scale):

                    r, g, b = matrix[y][x]

                    values_r.append(r)
                    values_g.append(g)
                    values_b.append(b)

            # Menghitung median setiap channel
            median_r = calculate_median(values_r)
            median_g = calculate_median(values_g)
            median_b = calculate_median(values_b)

            # Pixel hasil
            new_pixel = (
                median_r,
                median_g,
                median_b
            )

            row.append(new_pixel)

        output_matrix.append(row)

    return output_matrix

Penerapan pada gambar:

In [ ]:
downsampling.run_downsampling("median", scale)

# 4. Upsampling Algorithm

Load image hasil downsampling:

In [ ]:
# 1. Inisialisasi Upsampling dari Gambar Asli
upsampling_original = Sampling(images, matrices)

# 2. Inisialisasi Upsampling dari Hasil Downsampling (Avg, Max, Med)
image_downsampling_paths = {
    "image_1_avg": "/content/drive/MyDrive/PCD_Assignment01/results/downsampling/image_1_average.png",
    "image_1_max": "/content/drive/MyDrive/PCD_Assignment01/results/downsampling/image_1_max.png",
    "image_1_med": "/content/drive/MyDrive/PCD_Assignment01/results/downsampling/image_1_median.png",
    "image_2_avg": "/content/drive/MyDrive/PCD_Assignment01/results/downsampling/image_2_average.png",
    "image_2_max": "/content/drive/MyDrive/PCD_Assignment01/results/downsampling/image_2_max.png",
    "image_2_med": "/content/drive/MyDrive/PCD_Assignment01/results/downsampling/image_2_median.png",
}

images_ds = {}
matrices_ds = {}
for name, path in image_downsampling_paths.items():
    if os.path.exists(path):
        img = Image.open(path)
        images_ds[name] = img
        matrices_ds[name] = image_to_matrix(img)
    else:
        print(f"Warning: File tidak ditemukan di {path}")

upsampling_from_ds = Sampling(images_ds, matrices_ds)

print(f"Objek upsampling siap digunakan dengan {len(images_ds)} gambar hasil downsampling.")

Fungsi untuk menemukan piksel tetangga:

In [ ]:
def get_original_neighbors(matrix, y_out, x_out, scale, radius):
    input_height = len(matrix)
    input_width = len(matrix[0])

    neighbors = []

    y_min = max(
        0,
        (y_out - radius + scale - 1) // scale
    )

    y_max = min(
        input_height - 1,
        (y_out + radius) // scale
    )

    x_min = max(
        0,
        (x_out - radius + scale - 1) // scale
    )

    x_max = min(
        input_width - 1,
        (x_out + radius) // scale
    )

    for y_in in range(y_min, y_max + 1):
        for x_in in range(x_min, x_max + 1):

            # Posisi pixel asli pada output
            y_original = y_in * scale
            x_original = x_in * scale

            if (
                abs(y_original - y_out) <= radius
                and
                abs(x_original - x_out) <= radius
            ):
                neighbors.append(
                    matrix[y_in][x_in]
                )

    return neighbors

## 4.1 Nearest Neighbor Upsampling

In [ ]:
def nn_upsampling(matrix, scale, mode="min"):
    input_height = len(matrix)
    input_width = len(matrix[0])

    output_height = input_height * scale
    output_width = input_width * scale

    output_matrix = []

    for y_out in range(output_height):
        row = []

        for x_out in range(output_width):

            # Posisi piksel asli
            if (
                y_out % scale == 0
                and
                x_out % scale == 0
            ):

                y_in = y_out // scale
                x_in = x_out // scale

                pixel = matrix[y_in][x_in]

            else:

                neighbors = get_original_neighbors(
                    matrix,
                    y_out,
                    x_out,
                    scale,
                    radius=1
                )

                # Pisahkan channel RGB
                values_r = [pixel[0] for pixel in neighbors]
                values_g = [pixel[1] for pixel in neighbors]
                values_b = [pixel[2] for pixel in neighbors]

                if mode == "min":

                    new_r = min(values_r)
                    new_g = min(values_g)
                    new_b = min(values_b)

                elif mode == "max":

                    new_r = max(values_r)
                    new_g = max(values_g)
                    new_b = max(values_b)

                else:
                    raise ValueError(
                        "mode harus 'min' atau 'max'"
                    )

                pixel = (
                    new_r,
                    new_g,
                    new_b
                )

            row.append(pixel)

        output_matrix.append(row)

    return output_matrix

In [ ]:
print("Source: Original")
upsampling_original.run_upsampling("nearest neighbor", scale=2, source_name="original")

print("\nSource: Downsampled")
upsampling_from_ds.run_upsampling("nearest neighbor", scale=2, source_name="downsampled")

## 4.2 Bilinear Upsampling

Definisi fungsi:

In [ ]:
def bilinear_upsampling(matrix, scale):

    input_height = len(matrix)
    input_width = len(matrix[0])

    output_height = input_height * scale
    output_width = input_width * scale

    output_matrix = []

    for y_out in range(output_height):

        row = []

        for x_out in range(output_width):

            if (y_out % scale == 0 and x_out % scale == 0):
                y_in = y_out // scale
                x_in = x_out // scale
                pixel = matrix[y_in][x_in]
            else:
                neighbors = get_original_neighbors(
                    matrix,
                    y_out,
                    x_out,
                    scale,
                    radius=1
                )

                values_r = [pixel[0] for pixel in neighbors]
                values_g = [pixel[1] for pixel in neighbors]
                values_b = [pixel[2] for pixel in neighbors]

                new_r = round(
                    sum(values_r) / len(values_r)
                )

                new_g = round(
                    sum(values_g) / len(values_g)
                )

                new_b = round(
                    sum(values_b) / len(values_b)
                )

                pixel = (
                    new_r,
                    new_g,
                    new_b
                )

            row.append(pixel)

        output_matrix.append(row)

    return output_matrix

In [ ]:
print("Source: Original")
upsampling_original.run_upsampling("bilinear", scale=2, source_name="original")

print("\nSource: Downsampled")
upsampling_from_ds.run_upsampling("bilinear", scale=2, source_name="downsampled")

## 4.3 Bicubic Upsampling

In [ ]:
# Bobot neighborhood
weight_circle_1 = 0.7
weight_circle_2 = 0.3

In [ ]:
def bicubic_upsampling(
    matrix,
    scale,
    weight_circle_1=0.7,
    weight_circle_2=0.3
):

    # Pastikan total bobot = 1
    if round(weight_circle_1 + weight_circle_2, 10) != 1:
        raise ValueError(
            "Total bobot harus sama dengan 1."
        )

    input_height = len(matrix)
    input_width = len(matrix[0])

    output_height = input_height * scale
    output_width = input_width * scale

    output_matrix = []

    for y_out in range(output_height):

        row = []

        for x_out in range(output_width):
            if (y_out % scale == 0 and x_out % scale == 0):
                y_in = y_out // scale
                x_in = x_out // scale

                pixel = matrix[y_in][x_in]
            else:
                # Tetangga lingkar 1 (3x3)
                neighbors_1 = get_original_neighbors(
                    matrix,
                    y_out,
                    x_out,
                    scale,
                    radius=1
                )

                values_r_1 = [
                    pixel[0]
                    for pixel in neighbors_1
                ]

                values_g_1 = [
                    pixel[1]
                    for pixel in neighbors_1
                ]

                values_b_1 = [
                    pixel[2]
                    for pixel in neighbors_1
                ]

                avg_r_1 = sum(values_r_1) / len(values_r_1)
                avg_g_1 = sum(values_g_1) / len(values_g_1)
                avg_b_1 = sum(values_b_1) / len(values_b_1)

                # Tetangga lingkar 2 (5×5)
                neighbors_2 = get_original_neighbors(
                    matrix,
                    y_out,
                    x_out,
                    scale,
                    radius=2
                )

                values_r_2 = [
                    pixel[0]
                    for pixel in neighbors_2
                ]

                values_g_2 = [
                    pixel[1]
                    for pixel in neighbors_2
                ]

                values_b_2 = [
                    pixel[2]
                    for pixel in neighbors_2
                ]

                avg_r_2 = sum(values_r_2) / len(values_r_2)
                avg_g_2 = sum(values_g_2) / len(values_g_2)
                avg_b_2 = sum(values_b_2) / len(values_b_2)

                # Weighted average
                new_r = (weight_circle_1 * avg_r_1 + weight_circle_2 * avg_r_2)
                new_g = (weight_circle_1 * avg_g_1 + weight_circle_2 * avg_g_2)

                new_b = (weight_circle_1 * avg_b_1 + weight_circle_2 * avg_b_2)

                # Batasi RGB
                new_r = round(max(0, min(255, new_r)))
                new_g = round(max(0, min(255, new_g)))
                new_b = round(max(0, min(255, new_b)))

                pixel = (new_r, new_g, new_b)

            row.append(pixel)

        output_matrix.append(row)

    return output_matrix

In [ ]:
print("Source: Original")
upsampling_original.run_upsampling("bicubic", scale=2, source_name="original")

print("\nSource: Downsampled")
upsampling_from_ds.run_upsampling("bicubic", scale=2, source_name="downsampled")